# Chapter 12 — Support Vector Machines II: Maximum Margin, Soft Margin, Kernels & Multiclass SVMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch12_support_vector_machines_ii/ch12_notebook.ipynb)

Companion notebook for **Chapter 12** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- An SVM picks the separating hyperplane with the largest margin (1/‖w‖) to the nearest points.
- Only the support vectors matter: w = Σ αₙyₙxₙ, and every other point could be removed.
- A soft margin allows some violations, measured by the hinge loss. C trades margin width against training errors.
- Kernels make SVMs nonlinear: map to a higher-dimensional space, then separate with a hyperplane there.
- The kernel trick computes K(x, z) = Φ(x)ᵀΦ(z) without building Φ. Common kernels are linear, polynomial and Gaussian (RBF).
- For C classes, use one-against-the-rest (C SVMs) or one-against-one (C(C−1)/2 SVMs).

## Example 1: The Maximum-Margin Hyperplane

This uses the same data as the perceptron in Chapter 11, now with a linear SVM with a huge C (effectively a hard margin). Only 2 of the 40 points become support vectors. Both have y·g(x) = 1.000, which means they sit exactly on the margin. Rebuilding w from those two points alone, as Σ αₙyₙxₙ, gives the same w. The last line is the payoff: the closest point is 0.351 units from the SVM boundary, against only 0.032 for the perceptron's line. That is more than ten times the safety gap.

In [1]:
# Chapter 12 - Example 1: Maximum-margin hyperplane (hard-margin linear SVM)
# Same data as the perceptron example, so we can compare the two lines.
import numpy as np
from sklearn.svm import SVC

rng = np.random.default_rng(1)
X = np.vstack([rng.normal([1, 1], 0.5, (20, 2)), rng.normal([-1, -1], 0.5, (20, 2))])
y = np.r_[np.ones(20), -np.ones(20)]

svm = SVC(kernel="linear", C=1e6).fit(X, y)          # huge C = (almost) hard margin
w, w0 = svm.coef_[0], svm.intercept_[0]
print(f"SVM boundary: {w[0]:.3f}*x1 + {w[1]:.3f}*x2 + {w0:.3f} = 0")
print(f"Number of support vectors: {len(svm.support_)} (out of {len(X)} points)")
for sv, a in zip(svm.support_vectors_, svm.dual_coef_[0]):
    g = sv @ w + w0
    print(f"  support vector {np.round(sv, 3)}  y*g(x) = {np.sign(a) * g:.3f}  "
          f"alpha*y = {a:+.3f}")
print(f"\nMargin (distance from boundary to nearest point) = 1/||w|| = {1 / np.linalg.norm(w):.3f}")
print(f"Width of the empty 'street' between the classes  = 2/||w|| = {2 / np.linalg.norm(w):.3f}")

# Rebuild w from the support vectors only: w = sum(alpha_n * y_n * x_n)
w_from_sv = (svm.dual_coef_[0][:, None] * svm.support_vectors_).sum(axis=0)
print("w rebuilt from support vectors:", np.round(w_from_sv, 3), "(matches w)")

# Compare with the perceptron line learned in Chapter 11 on this exact data
w_perc = np.array([0.9, 0.066, 1.635])              # [w0, w1, w2] from Chapter 11, Example 2
d_perc = np.abs(np.column_stack([np.ones(40), X]) @ w_perc).min() / np.linalg.norm(w_perc[1:])
print(f"\nClosest point to the PERCEPTRON line: {d_perc:.3f}  vs  SVM: {1 / np.linalg.norm(w):.3f}")

SVM boundary: 1.411*x1 + 2.475*x2 + 1.364 = 0
Number of support vectors: 2 (out of 40 points)
  support vector [-0.703 -0.554]  y*g(x) = 1.000  alpha*y = -4.059
  support vector [-0.356  0.055]  y*g(x) = 1.000  alpha*y = +4.059

Margin (distance from boundary to nearest point) = 1/||w|| = 0.351
Width of the empty 'street' between the classes  = 2/||w|| = 0.702
w rebuilt from support vectors: [1.411 2.475] (matches w)

Closest point to the PERCEPTRON line: 0.032  vs  SVM: 0.351


## Example 2: Soft Margin and the C Parameter

Two overlapping classes, so a soft margin is needed. As C grows from 0.001 to 100, the margin shrinks from 4.03 to 0.55 and the number of support vectors falls from 178 to 36. The model trusts fewer points and fits the training set harder. Test accuracy stays about the same, which shows that a bigger C is not automatically better. The last lines sort the training points by hinge loss: outside the margin, inside it, or on the wrong side.

In [2]:
# Chapter 12 - Example 2: Soft margin - the C hyperparameter and the hinge loss
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
X = np.vstack([rng.normal([0, 0], 1.0, (150, 2)), rng.normal([2, 2], 1.0, (150, 2))])
y = np.r_[-np.ones(150), np.ones(150)]              # classes overlap a little
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)

print("    C     margin  support vectors  train acc  test acc  points with xi>1")
for C in (0.001, 0.01, 0.1, 1, 100):
    m = SVC(kernel="linear", C=C).fit(X_tr, y_tr)
    w, w0 = m.coef_[0], m.intercept_[0]
    xi = np.maximum(0, 1 - y_tr * (X_tr @ w + w0))  # hinge loss of each training point
    print(f"  {C:<7g}  {1 / np.linalg.norm(w):5.2f}       {len(m.support_):3d}          "
          f"{m.score(X_tr, y_tr):.3f}     {m.score(X_te, y_te):.3f}       {np.sum(xi > 1):3d}")
print("\nSmall C: wide margin, many support vectors, tolerates errors.")
print("Large C: narrow margin, tries hard to classify every training point.")

# What the hinge loss means for three individual points (C = 1 model)
m = SVC(kernel="linear", C=1).fit(X_tr, y_tr)
g = X_tr @ m.coef_[0] + m.intercept_[0]
xi = np.maximum(0, 1 - y_tr * g)
for label, mask in (("outside the margin (xi = 0)", xi == 0),
                    ("inside the margin (0 < xi < 1)", (xi > 0) & (xi < 1)),
                    ("wrong side (xi > 1)", xi > 1)):
    print(f"  {label:32s}: {mask.sum():3d} points")

    C     margin  support vectors  train acc  test acc  points with xi>1
  0.001     4.03       178          0.911     0.900        16
  0.01      1.52        97          0.917     0.917        15
  0.1       0.91        51          0.928     0.900        13
  1         0.64        39          0.928     0.900        13
  100       0.55        36          0.928     0.908        13

Small C: wide margin, many support vectors, tolerates errors.
Large C: narrow margin, tries hard to classify every training point.
  outside the margin (xi = 0)     : 143 points
  inside the margin (0 < xi < 1)  :  24 points
  wrong side (xi > 1)             :  13 points


## Example 3: Kernels and Multiclass SVMs

First, the kernel trick in numbers: (x·z + 1)² gives 25, and building the 6-dimensional Φ(x) and Φ(z) explicitly and taking their dot product also gives 25. Same answer, less work. Next, two rings of points that no line can separate. A linear SVM manages only 54%, while polynomial and Gaussian kernels both reach 100%. Finally, a 4-class problem: one-against-the-rest trains 4 SVMs, one-against-one trains 6 (= 4×3/2), and both reach similar accuracy.

In [3]:
# Chapter 12 - Example 3: Kernels and multiclass SVMs
import numpy as np
from sklearn.svm import SVC
from sklearn.datasets import make_circles, make_blobs
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.model_selection import train_test_split

# 1. The kernel trick: K(x, z) = (x.z + 1)^2 equals phi(x).phi(z) without building phi
x, z = np.array([1.0, 2.0]), np.array([3.0, 0.5])
phi = lambda v: np.array([1, v[0]**2, v[1]**2, np.sqrt(2)*v[0]*v[1],
                          np.sqrt(2)*v[0], np.sqrt(2)*v[1]])
print(f"Kernel (x.z + 1)^2      = {(x @ z + 1) ** 2:.1f}")
print(f"phi(x) . phi(z) (6-dim) = {phi(x) @ phi(z):.1f}   <- same number, less work")

# 2. Linear vs polynomial vs Gaussian (RBF) kernel on two rings
X, y = make_circles(n_samples=400, factor=0.4, noise=0.08, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
print("\nTwo rings (not linearly separable):")
for name, m in (("linear", SVC(kernel="linear")),
                ("polynomial (p=2)", SVC(kernel="poly", degree=2, coef0=1)),
                ("Gaussian / RBF", SVC(kernel="rbf", gamma=1))):
    m.fit(Xtr, ytr)
    print(f"  {name:17s} test accuracy {m.score(Xte, yte):.3f}   support vectors {len(m.support_)}")

# 3. Multiclass with binary SVMs: one-against-the-rest vs one-against-one
X, y = make_blobs(n_samples=600, centers=4, cluster_std=1.8, random_state=4)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
ovr = OneVsRestClassifier(SVC(kernel="rbf")).fit(Xtr, ytr)
ovo = OneVsOneClassifier(SVC(kernel="rbf")).fit(Xtr, ytr)
C = 4
print(f"\n{C} classes:")
print(f"  One-against-the-rest: {len(ovr.estimators_)} SVMs, test accuracy {ovr.score(Xte, yte):.3f}")
print(f"  One-against-one     : {len(ovo.estimators_)} SVMs (C(C-1)/2 = {C*(C-1)//2}), "
      f"test accuracy {ovo.score(Xte, yte):.3f}")

Kernel (x.z + 1)^2      = 25.0
phi(x) . phi(z) (6-dim) = 25.0   <- same number, less work

Two rings (not linearly separable):
  linear            test accuracy 0.542   support vectors 274
  polynomial (p=2)  test accuracy 1.000   support vectors 16
  Gaussian / RBF    test accuracy 1.000   support vectors 34

4 classes:
  One-against-the-rest: 4 SVMs, test accuracy 0.878
  One-against-one     : 6 SVMs (C(C-1)/2 = 6), test accuracy 0.894
